# RAG Workshop #2 : Embedding Strategyの研究
- Embedding Strategy 1 ~ 4

## Embedding Strategy 1: Coarse (粗い) chunk, Granular (小さな) chunk で Vectorize

In [29]:
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import chromadb

from langchain_text_splitters import RecursiveCharacterTextSplitter


embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="./my_chroma_db")

vector_db = Chroma(
    client=chroma_client,
    collection_name="tokyo_info",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

### VectorDBを初期化

In [30]:
tokyo_granular_collection = Chroma(
    collection_name="tokyo_granular", embedding_function=embedding_model
)

tokyo_granular_collection.reset_collection()


tokyo_coarse_collection = Chroma(
    collection_name="tokyo_coarse", embedding_function=embedding_model
)

tokyo_coarse_collection.reset_collection()

In [31]:
from langchain_community.document_loaders import AsyncHtmlLoader

destination_url = "https://en.wikipedia.org/wiki/Tokyo"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()

len(docs)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.48it/s]


1

### Granular (小さな) chunk

#### HTML Splitterを設定
- `HTMLSectionSplitter`を使用

In [32]:
from langchain_text_splitters import HTMLSectionSplitter

headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)


def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(html_string)
        all_chunks.extend(temp_chunks)

    return all_chunks

#### granular chunk を設定 VectorDB にgranular chunkをingest

In [33]:
granular_chunks = split_docs_into_granular_chunks(docs)
len(granular_chunks)
tokyo_granular_collection.add_documents(documents=granular_chunks)

['2dae6a0c-ac8c-4ca0-bfb6-7ffdac6d895f',
 '144d5baa-8e5a-44fe-b691-c4fba4fca3be',
 '4f5e9a55-8bdf-4924-b2d5-57ea051199d9',
 'fda46db6-8e1c-4e4d-b3a3-8285fc611d96',
 'c84e3367-962c-412d-8a4b-272efe865497',
 '13feae1b-49d2-4898-8e33-0cfad6e16b54',
 '2efb7af2-3af0-4b2e-9b15-f22487c89b36',
 'a15d8bcf-94b6-410e-b75c-6dbeaafcdb08',
 '5c5d21fb-b06c-4e02-8537-7b66aa228ebb',
 '576cab2e-d41b-431a-95ae-95ebe1b5fb8f',
 'd294ba28-3393-4a35-bdee-f0b39a99e371',
 'a57c9e00-ea31-4be2-a8ae-a1fb25cd253d',
 '6c7b6219-ce18-4cb9-ba55-8b95599e7124',
 '9b65d3ce-d5cd-46b1-9a43-83530d36f026',
 'ed3a5703-95b4-4793-8eb1-8b37def0f809',
 '829e7c57-e4ea-4e27-b69d-f2c544af79a5',
 '528491b5-8dc3-42a9-97a3-5ce945307ff8',
 '92d7f5ac-6b9d-4ec4-975e-de46c8edccb9',
 '512ebc47-51e2-4d7e-80e5-3d96f4b94cf1']

#### granular chunkを検索

In [34]:
results = tokyo_granular_collection.similarity_search(
    query="Famous places in Tokyo", k=3
)
for idx, doc in enumerate(results):
    print("=========" * 20)
    print(f"🍅 Document {idx + 1}:")
    print("=========" * 20)
    print("\n")
    print(doc)

🍅 Document 1:


page_content='Further reading 
 Guides 
 
 Bender, Andrew, and Timothy N. Hornyak.  Tokyo  (City Travel Guide) (2010) 
 Mansfield, Stephen.  Dk Eyewitness Top 10 Travel Guide: Tokyo  (2013) 
 Waley, Paul.  Tokyo Now and Then: An Explorer's Guide . (1984). 592 pp 
 Yanagihara, Wendy.  Lonely Planet Tokyo Encounter 
 
 
 Contemporary 
 
 Allinson, Gary D.  Suburban Tokyo: A Comparative Study in Politics and Social Change . (1979). 258 pp. 
 Bestor, Theodore.  Neighborhood Tokyo  (1989).  online edition 
 
 Bestor, Theodore.  Tsukiji: The Fish Market at the Centre of the World . (2004)  online edition [ permanent dead link ] 
 
 Fowler, Edward.  San'ya Blues: Labouring Life in Contemporary Tokyo . (1996)  
 ISBN   0-8014-8570-3 . 
 Friedman, Mildred, ed.  Tokyo, Form and Spirit . (1986). 256 pp. 
 Jinnai, Hidenobu.  Tokyo: A Spatial Anthropology . (1995). 236 pp. 
 Jones, Sumie et al. eds.  A Tokyo Anthology: Literature from Japan's Modern Metropolis, 1850–1920  (2017); pr

### coarse (粗い) chunk

#### Splitterを設定
- `RecursiveCharacterTextSplitter`を使用

In [35]:
from langchain_community.document_transformers import Html2TextTransformer

html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)


def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    coarse_chunks = text_splitter.split_documents(text_docs)

    return coarse_chunks

#### coarse chunkをVectorDBにingest

In [37]:
coarse_chunks = split_docs_into_coarse_chunks(docs)
len(coarse_chunks)
tokyo_coarse_collection.add_documents(documents=coarse_chunks)

['c4ada509-7860-4083-b0f3-2fde7290f0ba',
 '903395fe-41ca-4ee3-9081-56049c8c9700',
 '001fd07d-b5d9-4572-8a74-fb9f641c85f5',
 '78f51a8f-c9cc-418d-b3d1-d64fd92c83ad',
 '68bebf30-dbf4-45f4-a509-9be6f1b0c1eb',
 'b0829f0c-528c-4856-9977-b60d242b1255',
 '5fc0f9f4-ef25-4c3d-941b-8e8ffcf92cfc',
 'b0f7cbbd-8c1a-4a79-b0f5-7ca52c687e0c',
 'e22b3dac-d019-4190-9ff2-bca4fa5dad65',
 '3e8fe1e1-d472-418d-9a93-56d1b53a8678',
 '3076b43b-5c64-4228-85c5-7a25f0c10e32',
 'ddeee3a7-ef55-4650-a4d9-a7754e1903c1',
 '053c007b-5f5e-40cc-b887-a548f4099b77',
 'b22a2a21-9807-48b1-be81-3af991cdeb31',
 '33366d50-dae3-45cd-8ba7-694641308b89',
 '6c48865e-be8d-4f6e-b5f1-afe8fa9f067d',
 '8267d37f-a2c4-4ef4-8a81-dc279615d8e9',
 '14b89baa-584d-464e-af30-27f6dc7f12ce',
 '4ec73f73-9673-42cc-93bf-c28389adfd45',
 '1770a297-882b-4424-b017-12f42d08642c',
 '0c376f8f-bbfb-46bf-a42c-74ee7900407c',
 '070367c6-4cd5-455a-bc18-84e3213be934',
 '94753a17-c6b8-43e0-8e79-9274aeb48c33',
 '628f6168-198c-41fc-8517-7c3e95ff41bd',
 'f23a5f41-e972-

#### coarse chunkを検索

In [38]:
results = tokyo_coarse_collection.similarity_search(query="Famous places in Tokyo", k=3)

for idx, doc in enumerate(results):
    print("=========" * 20)
    print(f"🍅 Document {idx + 1}:")
    print("=========" * 20)
    print("\n")
    print(doc)

🍅 Document 1:


page_content='Articles related to Tokyo  
---  
| Preceded byHeian-kyō | **Capital of Japan **  
1868–present  | **Most recent**  
---|---|---  
  
  * v
  * t
  * e

Tokyo Metropolis  
---  
  
  * Architecture
  * Education
  * Festivals
  * History
  * Neighborhoods
  * Politics
  * Sports
  * Symbols
  * Tourism
  * Transportation

  
Special Wards  
of Tokyo|

  * Adachi
  * Arakawa
  * Bunkyō
  * Chiyoda
  * Chūō
  * Edogawa
  * Itabashi
  * Katsushika
  * Kita
  * Kōtō
  * Meguro
  * Minato
  * Nakano
  * Nerima
  * Ōta
  * Setagaya
  * Shibuya
  * Shinagawa
  * Shinjuku
  * Suginami
  * Sumida
  * Taitō
  * Toshima

  
Western  
(Tama area)| | Core city| 

  * Hachiōji

  
---|---  
Cities|

  * Akiruno
  * Akishima
  * Chōfu
  * Fuchū
  * Fussa
  * Hamura
  * Higashikurume
  * Higashimurayama
  * Higashiyamato
  * Hino
  * Inagi
  * Kiyose
  * Kodaira
  * Koganei
  * Kokubunji
  * Komae
  * Kunitachi
  * Machida
  * Mitaka
  * Musashimurayama
  * Musashino
  * 

### WikivoyageにあるTokyoにあるさまざまな地点で Vectorize -> Ingest -> 検索

#### urlリストを作成

In [39]:
tokyo_cities = [
    "Tokyo",
    "Tokyo/Chiyoda",
    "Tokyo/Chuo",
    "Tokyo/Minato",
    "Tokyo/Shinjuku",
    "Tokyo/Shibuya",
    "Tokyo/Shinagawa",
    "Tokyo/Toshima",
    "Tokyo/Meguro",
    "Tokyo/Sumida",
    "Tokyo/Taio",
    "Tokyo/Bunkyo",
    "Tokyo/East",
    "Tokyo/North",
    "Tokyo/Nakano",
    "Tokyo/Ota",
    "Tokyo/Setagaya",
    "Tokyo/Suginami",
]

wikipedia_root_url = "https://en.wikivoyage.org/wiki"
tokyo_destination_urls = [f"{wikipedia_root_url}/{d}" for d in tokyo_cities]

#### VectorDBにingest
- granular、coarseに分けてingest

In [40]:
for destination_url in tokyo_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()

    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        tokyo_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        tokyo_coarse_collection.add_documents(documents=coarse_chunks)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.82it/s]


{'source': 'https://en.wikivoyage.org/wiki/Tokyo', 'title': 'Tokyo – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.35it/s]


{'source': 'https://en.wikivoyage.org/wiki/Tokyo/Chiyoda', 'title': 'Tokyo/Chiyoda – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.41it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Tokyo/Ota', 'title': 'Tokyo/Ota – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.45it/s]


{'source': 'https://en.wikivoyage.org/wiki/Tokyo/Setagaya', 'title': 'Tokyo/Setagaya – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.45it/s]


{'source': 'https://en.wikivoyage.org/wiki/Tokyo/Suginami', 'title': 'Tokyo/Suginami – Travel guide at Wikivoyage', 'language': 'en'}


#### granular chunk 検索

In [46]:
granular_results = tokyo_granular_collection.similarity_search(
    query="Events or festivals in East Tokyo", k=4
)
for idx, doc in enumerate(granular_results):
    print("=========" * 20)
    print(f"🍅 Document {idx + 1}:")
    print("=========" * 20)
    print("\n")
    print(doc)

🍅 Document 1:


page_content='Eastern Tokyo' metadata={'Header 1': 'Eastern Tokyo'}
🍅 Document 2:


page_content='Tokyo' metadata={'Header 1': 'Tokyo'}
🍅 Document 3:


page_content='Understand 
 [ edit ] 
 
 Sumida is considered "shitamachi" (roughly translated as "old town"), though it has become a kind of bedroom community for Tokyoites, which has meant the building of many high-rise apartment buildings. Despite the boom in construction, the area retains its pre-WWII charm, with many small businesses and a small neighborhood feel to it. 
 The  Ryōgoku  (両国) neighborhood, in the southwest portion of the ward, is nearly synonymous with  sumō wrestling , one of Japan's most famous sports, where the human behemoths grapple and attempt to hoist each other out of the ring. The  Edo-Tokyo Museum , an excellent and large museum on the history of Tokyo (but closed for renovation until 2025), is here, as well as a collection of quirky special-interest museums. 
 
 Tourist information 
 [ edit 

#### coarse chunk 検索

In [42]:
coarse_results = tokyo_coarse_collection.similarity_search(
    query="Events or festivals in East Tokyo", k=4
)
for idx, doc in enumerate(coarse_results):
    print("=========" * 20)
    print(f"🍅 Document {idx + 1}:")
    print("=========" * 20)
    print("\n")
    print(doc)
    print("\n")

🍅 Document 1:


page_content='### Festivals

[edit]

  * **Sanja Matsuri** (三社祭), third weekend in May. Tokyo's largest festival, held near Sensoji Temple in Asakusa, this three-day extravaganza sees up to 2 million people turn out to watch the parade of portable shrines (_mikoshi_) with music, dancing and geisha performances.
  * **Sumidagawa Fireworks Festival** (隅田川花火大会 _Sumidagawa Hanabi Taikai_), fourth Saturday in July. Huge fireworks competition that sees up to a million people line the banks of the Sumida River.

## Learn

[edit]

The curious can study traditional culture such as **tea ceremony** ,
**calligraphy** , or **martial arts** such as Karate, Judo, Aikido and Kendo.
There are also many language schools to help you work on your Japanese.
Several universities in Tokyo cater to international students at the
undergraduate or graduate level.

### Universities

[edit]

  * **Keio University** (慶應義塾大学 _Keiō Gijuku Daigaku_) — Japan's top private university (unless you ask a W

## Embedding Strategy 2: `PrentDocumentRetriever`を使用

In [47]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### parent, child Splitterを設定
- chunk_sizeをそれぞれ3000, 500で設定

In [48]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)


child_chunks_collection = Chroma(
    collection_name="tokyo_child_chunks", embedding_function=embedding_model
)

child_chunks_collection.reset_collection()

doc_store = InMemoryStore()

parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

### VectorDBにingest

In [49]:
for destination_url in tokyo_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    print(f"Ingesting {destination_url}")
    parent_doc_retriever.add_documents(list(text_docs), ids=None)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.79it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.35it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Chiyoda


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.50it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Tokyo/Ota


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.49it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Setagaya


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.64it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Suginami


### 検索

In [ ]:
from IPython.display import Markdown, display

query3 = "Tower Records"
retrieved_docs = parent_doc_retriever.invoke(query3)
print(f"length of retrieved docs: {len(retrieved_docs)}")
display(Markdown(retrieved_docs[0].page_content))

length of retrieved docs: 4


### Music

[edit]

  * Disk Union, Shinjuku 3-34-1 (Main Branch is near Shinjuku Sanchome Station), ☏ +81 3-3352-2697. You can get music, movies, and music books. Another branch at Shinjuku 3-17-5 specialize in specific genres or used goods. Great for music enthusiasts.
  * HMV (Basement floor of Lumine Est), ☏ +81 3-5269-2571. HMV is one of the bigger record stores in Japan with a good selection of music and movies. Another store at Shinjuku Alta 6F. (updated Dec 2023)
  * Tower Records, Shinjuku 3-37-1 (Southeast Exit of the JR Shinjuku Station, 9-10F of Flags building), ☏ +81 3-5360-7811. Tower Records is one of the biggest record stores in Japan. They have any CD or DVD you can imagine, and if not, you can probably order or reserve it.
  * Nishi-Shinjuku 7-chome (northwest of JR Shinjuku station). Packed with music shops specializing in various genres such as punk and heavy metal. Many sell nothing but bootlegs and collectibles.

### Other

[edit]

  * West Exit Square Event Space (西口広場イベントスペース) (Between the underground entrance to the Keio department store and the taxi rotary). An area hosting a rotating series of stalls or exhibits. These have included various local foods from around Japan, furniture, art prints and information about various government projects around Tokyo. The underground area around it is a great place to find takeaway food and fast food outlets - named Keio Mall, Odakyu Ace and Shinjuku Delish Park (temporary location for Odakyu's depachika). (updated Mar 2024)

## Eat

[edit]

A great way to get by in Tokyo on a budget is to make lunch your main meal.
Many restaurants cater to the business lunch crowd and offer an excellent two
or three course meal for between ¥800-1300 or lunch buffet for \1000-2000.
Going to the same places for dinner would be up to three times more expensive.

Shinjuku has more than 5,000 eateries, the most among the 23 special wards of
Tokyo.

### Budget

[edit]

### 検索

In [56]:
child_docs_only = child_chunks_collection.similarity_search(query3)
print(f"length of child docs only: {len(child_docs_only)}")

display(Markdown(child_docs_only[0].page_content))

length of child docs only: 4


* Tower Records, Shinjuku 3-37-1 (Southeast Exit of the JR Shinjuku Station, 9-10F of Flags building), ☏ +81 3-5360-7811. Tower Records is one of the biggest record stores in Japan. They have any CD or DVD you can imagine, and if not, you can probably order or reserve it.
  * Nishi-Shinjuku 7-chome (northwest of JR Shinjuku station). Packed with music shops specializing in various genres such as punk and heavy metal. Many sell nothing but bootlegs and collectibles.

## Embedding Strategy 3: `MultiVectorRetriever`を使用

In [57]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma

from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### `RecursiveCharacterTextSplitter`でparent, chiild splitterをそれぞれ設定

In [58]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunks_collection = Chroma(
    collection_name="tokyo_child_chunks",
    embedding_function=embedding_model,
)

child_chunks_collection.reset_collection()
doc_byte_store = InMemoryByteStore()
doc_store = InMemoryStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    byte_store=doc_byte_store,
)

### urlごとにingest

In [59]:
for destination_url in tokyo_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    coarse_chunks = parent_splitter.split_documents(text_docs)

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[i]

        granular_chunks = child_splitter.split_documents([coarse_chunk])

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id

        all_granular_chunks.extend(granular_chunks)

    print(f"Ingesting {destination_url}")
    multi_vector_retriever.vectorstore.add_documents(all_granular_chunks)
    multi_vector_retriever.docstore.mset(list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.57it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.34it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Chiyoda


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.54it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Tokyo/Ota


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.49it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Setagaya


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.49it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Suginami


### 検索

In [60]:
retrieved_docs = multi_vector_retriever.invoke("Tower Records")
print(f"length of retrieved docs: {len(retrieved_docs)}")

display(Markdown(retrieved_docs[0].page_content))

length of retrieved docs: 4


### Music

[edit]

  * Disk Union, Shinjuku 3-34-1 (Main Branch is near Shinjuku Sanchome Station), ☏ +81 3-3352-2697. You can get music, movies, and music books. Another branch at Shinjuku 3-17-5 specialize in specific genres or used goods. Great for music enthusiasts.
  * HMV (Basement floor of Lumine Est), ☏ +81 3-5269-2571. HMV is one of the bigger record stores in Japan with a good selection of music and movies. Another store at Shinjuku Alta 6F. (updated Dec 2023)
  * Tower Records, Shinjuku 3-37-1 (Southeast Exit of the JR Shinjuku Station, 9-10F of Flags building), ☏ +81 3-5360-7811. Tower Records is one of the biggest record stores in Japan. They have any CD or DVD you can imagine, and if not, you can probably order or reserve it.
  * Nishi-Shinjuku 7-chome (northwest of JR Shinjuku station). Packed with music shops specializing in various genres such as punk and heavy metal. Many sell nothing but bootlegs and collectibles.

### Other

[edit]

  * West Exit Square Event Space (西口広場イベントスペース) (Between the underground entrance to the Keio department store and the taxi rotary). An area hosting a rotating series of stalls or exhibits. These have included various local foods from around Japan, furniture, art prints and information about various government projects around Tokyo. The underground area around it is a great place to find takeaway food and fast food outlets - named Keio Mall, Odakyu Ace and Shinjuku Delish Park (temporary location for Odakyu's depachika). (updated Mar 2024)

## Eat

[edit]

A great way to get by in Tokyo on a budget is to make lunch your main meal.
Many restaurants cater to the business lunch crowd and offer an excellent two
or three course meal for between ¥800-1300 or lunch buffet for \1000-2000.
Going to the same places for dinner would be up to three times more expensive.

Shinjuku has more than 5,000 eateries, the most among the 23 special wards of
Tokyo.

### Budget

[edit]

In [61]:
child_docs_only = child_chunks_collection.similarity_search("Tower Records")
print(f"length of child docs only: {len(child_docs_only)}")
display(Markdown(child_docs_only[0].page_content))

length of child docs only: 4


* Tower Records, Shinjuku 3-37-1 (Southeast Exit of the JR Shinjuku Station, 9-10F of Flags building), ☏ +81 3-5360-7811. Tower Records is one of the biggest record stores in Japan. They have any CD or DVD you can imagine, and if not, you can probably order or reserve it.
  * Nishi-Shinjuku 7-chome (northwest of JR Shinjuku station). Packed with music shops specializing in various genres such as punk and heavy metal. Many sell nothing but bootlegs and collectibles.

## Embedding Strategy 4: `MultiVectorRetriever`にsummaryをそれぞれのchunkに追加

In [62]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### `RecursiveCharacterTextSplitter`を使用してchunkを作成

In [63]:
from langchain_ollama import ChatOllama

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)

summaries_collection = Chroma(
    collection_name="tokyo_summaries",
    embedding_function=embedding_model,
)

summaries_collection.reset_collection()
doc_store = InMemoryStore()
doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(  # E
    vectorstore=summaries_collection, byte_store=doc_byte_store, docstore=doc_store
)

llm = ChatOllama(model="gemma3:1b", temperature=0)

### chunkをsummarizeするchainを設定

In [64]:
summarization_chain = (
    {"document": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Summarize the following document:\n\n{document}"
    )
    | llm
    | StrOutputParser()
)

### urlごとにingest(summarizeを含む)

In [65]:
for destination_url in tokyo_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    coarse_chunks = parent_splitter.split_documents(text_docs)

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[i]

        summary_text = summarization_chain.invoke(coarse_chunk)
        summary_doc = Document(
            page_content=summary_text, metadata={doc_key: coarse_chunk_id}
        )

        all_summaries.append(summary_doc)
        print(f"Summary for chunk {i + 1}/{len(coarse_chunks)} added.")

    print(f"Ingesting {destination_url}")
    multi_vector_retriever.vectorstore.add_documents(all_summaries)
    multi_vector_retriever.docstore.mset(list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.60it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.36it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Chiyoda


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.47it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Tokyo/Ota


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.39it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Setagaya


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.53it/s]


Ingesting https://en.wikivoyage.org/wiki/Tokyo/Suginami


### 検索

In [66]:
retrieved_docs = multi_vector_retriever.invoke("Tower Records")
print(f"length of retrieved docs: {len(retrieved_docs)}")
display(Markdown(retrieved_docs[0].page_content))

length of retrieved docs: 4


### Museums

[edit]

  * 35.636862139.7192696 Tokyo Metropolitan Teien Art Museum (東京都庭園美術館) (five minutes from the East exit of Meguro Station on Yamanote line). Art museum housed in a classic Art Deco style structure, connected to a small park with Japanese and Western gardens and outdoor sculptures.
  * 35.66216139.7176037 Nezu Museum, 6-5-1 Minamiaoyama, ☏ +81 3-3400-2536. Showcases the private art collection of Nezu Kaichirō with many pieces from the Edo period.
  * 35.661308139.7155838 Taro Okamoto Memorial Museum, 6-1-19 Minamiaoyama (8 minutes from Omotesando station on foot), ☏ +81 3-3406-0801. Former studio and house of Okamoto Taro. Unfinished works and paintings are shown. He made numerous masterpieces such as "Tower of the Sun" at the Expo 1970 and "Myth of the Future" mural now exhibited in Shibuya.
  * 35.673056139.7172229 TEPIA, 2 Chome-8-44 Kitaaoyama. Tu-Su 10:00-18:00, closed on holidays. Science museum showcasing new technologies, robots, home automation. Explanations are in Japanese but playing with the objects is still fun for kids and adults. Free. (updated Oct 2015)
  * 35.628284139.73266410 Aquapark shinagawa (アクアパーク品川) (two minutes walk from Takanawa exit of Shinagawa station). A popular aquarium with exhibits that combine dolphin shows and technology. The tunnel tank includes manta rays, green sawfish, and the only dwarf sawfish on display in the world.
  * Japan Traditional Crafts Aoyama Square (伝統工芸 青山スクエア), 8-1-22 Akasaka. 11:00-19:00. Gallery and shop for handicraft products from all over Japan. Free. (updated Mar 2024)

## Do

[edit]

  * Tokyo Cruise (水上バス 東京湾クルーズ (waterborne bus Tokyo Bay cruise)). Plies the Sumida River and Tokyo Bay between Hamamatsucho, Odaiba, Asakusa and other points. Fares vary depending on routing.

## Buy

[edit]

  * 35.653965139.762541 Tokyo Island (東京愛らんど Tōkyō Airando), 1-12-2 Kaigan (in Takeshiba Passenger Ship Terminal), ☏ +81 3-5472-6559. Daily 09:00-22:30. Antenna shop of Izu Isands and Ogasawara Islands.
  * 35.65648139.733652 Azabu-Jūban (麻布十番). A quieter commercial district to the southeast of Roppongi Hills, and a good place to spend a lazier afternoon browsing through shops and enjoying the local cafés. The surrounding residential area is popular among professional expats, so expect to see many international families as you walk through.
  * 35.669802139.7525033 Japan Sake and Shōchū Information Center (日本の酒情報館 nihon no sake jōhōkan), 1-1-21 Nishi-Shinbashi (10 min from Shimbashi station), ☏ +81 3-3519-2091. M-F 10:00-18:00, closed weekends and holidays. This four-story complex run by the Central Brewers' Union sells anything and everything related to sake, including cups, glasses, books, hydrometers and, of course, the nectar itself. \315/515 gets you a taste of 3/5 sakes that change daily.

## Eat

[edit]

### Budget

[edit]

Lunch boxes from supermarkets is another good option to save some money

In [70]:
summary_docs_only = summaries_collection.similarity_search("Tower Records")
print(f"length of summary docs only: {len(summary_docs_only)}")
display(Markdown(summary_docs_only[0].page_content))

length of summary docs only: 4


Okay, here’s a summary of the provided document, broken down into key points:

**Museums:**

*   **Tokyo Metropolitan Teien Art Museum:** A large, classic Art Deco museum housed in a beautiful park with Japanese and Western gardens.
*   **Nezu Museum:** Showcases the private art collection of Nezu Kaichirō, featuring Edo period pieces and a significant portion of his later works, including "Tower of the Sun" and "Myth of the Future."
*   **Taro Okamoto Memorial Museum:** A former studio and house where Taro Okamoto, a prominent artist, created many iconic works, including "Tower of the Sun" and "Myth of the Future."
*   **TEPIA:** A science museum focused on new technologies, robots, and home automation, with interactive exhibits for kids and adults.
*   **Aquapark Shinagawa:** A popular aquarium with manta rays, green sawfish, and a dwarf sawfish exhibit.

**Shopping & Activities:**

*   **Tokyo Island:** Offers a stroll along the Sumida River and Tokyo Bay, with a free aquarium (Aquapark Shinagawa) and a shopping mall (Tokyo Island).
*   **Japan Traditional Crafts Aoyama Square:** A gallery and shop showcasing handcrafted goods from Japan.
*   **Azabu-Jūban:** A quieter commercial district with a relaxed atmosphere, popular with expats.
*   **Japan Sake and Shōchū Information Center:** A four-story complex selling sake-related products, including cups, glasses, books, and nectar.

**Transportation:**

*   **Tokyo Cruise:** A boat tour along the Sumida River and Tokyo Bay.
*   **Tokyo Island Cruise:** A boat tour along the Sumida River and Tokyo Bay.

**Food & Drink:**

*   **Lunch Boxes:** A convenient and affordable way to eat.

**Overall:** The document provides a quick overview of various attractions and activities in Tokyo, covering art, culture, shopping, and leisure.

---

Do you want me to elaborate on any of these points, or perhaps focus on a specific category (e.g., transportation, shopping)?

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="./my_chroma_db")

vector_db = Chroma(
    client=chroma_client,
    collection_name="tokyo_info",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

In [5]:
wikipedia_loader = WikipediaLoader(query="Paestum")
wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
vector_db.add_documents(wikipedia_chunks)

['1935ffcb-726b-442a-a5ce-b83d03903fe8',
 '99eea114-d304-461c-b959-c0db201aacc6',
 '8c40a22f-e788-4e25-8f35-134902d1d061',
 '8482eb87-3c91-47fd-a289-694f180d7922',
 '07024954-430d-4196-96fe-5a666c6ad931',
 '255a9019-090c-45f5-85b6-9ec47b375e03',
 '11ab2b08-5ee0-46b6-b587-14ff069b899a',
 'ace31c57-111b-48c4-b27b-cab930d85655',
 'd2c05cea-9224-4d95-924f-cf44b4584bbb',
 '58c40ddb-2845-41fd-9b55-86849b976699',
 '71b00251-0e14-48b7-8b72-02f5055b6c9b',
 '8c2e16a3-d0d2-465d-9abb-c520921b6dd4',
 '4a2d5c62-3c6d-40a6-b2c0-b34d4c92080f',
 '86ce65be-0308-4f95-9525-c5c6638a7c03',
 '786e5289-6413-44f6-935b-ff1b1cfadad8',
 'e8f52bac-73f6-4258-9280-516da29dfb37',
 '71b59649-eb36-4fc2-a6fa-3a0d4f7be0df',
 'b27de6fd-1f3a-43c3-81c5-9ca26b976083',
 '0aed66b0-5302-4b93-9405-f9c6a4fe4d5c',
 '678906f1-af79-499c-b14b-285b5c7f38c6',
 '73da9cf9-63cc-43b3-aaab-48bbecbdf7c3',
 '646f427b-c15e-496b-be4d-88c933e331e4',
 'e47dfd83-ba6c-41dd-b0d6-2ebb18ff1714',
 '40417901-e28c-4e98-a8b6-046f2e09f217',
 '1eafa41b-9889-

In [7]:
word_loader = Docx2txtLoader(Path("./data/Paestum/Paestum-Britannica.docx"))
word_chunks = text_splitter.split_documents(word_loader.load())
vector_db.add_documents(word_chunks)

['92be61d3-e169-4a1c-8b84-0a72626fe207',
 'b1a47a45-0934-4c6b-b1ff-c4b1d747f375',
 'bdc3a1b9-d22d-4835-874c-8d25a2e0e882',
 'f2ec64a1-afd6-4a28-96b2-3eecac59bc72',
 'b63b483d-e3a4-400e-b36e-693e94c6c9d3',
 '907019cc-ef58-45e1-80a7-a726ab597ee0',
 '2022a82a-f6bd-46aa-a221-a99332dc6afb',
 '6b2e23ef-b5b1-442c-9c7c-0e1f6797d2c7']

In [9]:
pdf_loader = PyPDFLoader(Path("./data/Paestum/PaestumRevisited.pdf"))
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['ca06c034-545b-4ee7-9ce4-db6c5ba8869e',
 '91c238aa-19c4-4d81-98a2-8312773dc596',
 'af131616-be12-4fc3-a6ea-51403b183abc',
 '53dd19b1-bfe6-4f53-9635-d038c145030a',
 'e5c7829d-4ae5-4811-9965-df9c4a5889a6',
 '6d0c4472-ea32-4e55-8048-8a16edf06731',
 '0115e583-cbcc-4361-9e3a-e101b5cc0ecd',
 'bba2b900-ace0-4d8c-b079-05cdff72f304',
 'b23c8b11-4238-47df-94b7-5ae1eba7fdd3',
 '939c7739-c5ec-4fe0-8ea2-d837cf87e8ab',
 'cbf3801b-c046-4640-8f78-daf1e47a3d25',
 '8f2b39cb-d442-46a0-a061-681ae287935e',
 'bd6394b1-748f-4e0d-ba63-598fea34d101',
 '78aeb6ae-f6d4-499c-bd08-02ab871b1b00',
 '98e0fb74-4df9-41d1-a04f-da970bee1179',
 'cf1acac1-9dc8-48df-a752-718123f02844',
 '7237e53f-1504-4155-906e-18fb202f594c',
 '31b05505-b263-4aa1-a10e-a12a22536dcf',
 'cc270e49-5ae4-4f27-9b42-d766bb0bfad6',
 'b3592787-fa33-4e7d-9054-face08e3c6a6',
 'cbc3431a-3089-4f4b-9b69-0438d83d5e3c',
 'aa34544e-536e-4f2e-a3ab-22d1d052cf84',
 '0605f787-d86d-4abb-b299-769ebd1d5b90',
 '88f680c4-751a-443f-b497-73ce123b53bd',
 '8d5ad9ec-b83d-

In [10]:
txt_loader = TextLoader(Path("./data/Paestum/Paestum-Encyclopedia.txt"))
txt_chunks = text_splitter.split_documents(txt_loader.load())
vector_db.add_documents(txt_chunks)

['57bfaad4-cbef-4eca-8b3a-b09debd102f8']

## Data Cleaning:

In [11]:
def split_and_import(loader):
    chunks = text_splitter.split_documents(loader.load())
    vector_db.add_documents(chunks)
    print(f"Ingested chunks created by {loader}")

In [12]:
wikipedia_loader = WikipediaLoader(query="Paestum")
split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader(Path("./data/Paestum/Paestum-Britannica.docx"))
split_and_import(word_loader)

pdf_loader = PyPDFLoader(Path("./data/Paestum/PaestumRevisited.pdf"))
split_and_import(pdf_loader)

txt_loader = TextLoader(Path("./data/Paestum/Paestum-Encyclopedia.txt"))
split_and_import(txt_loader)

Ingested chunks created by <langchain_community.document_loaders.wikipedia.WikipediaLoader object at 0x16238ac00>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x10450e120>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x30c15c590>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x1056bc080>


## 複数のドキュメントをフォルダからIngest

### フォルダの中のファイルをすべてiterate

In [14]:
loader_classes = {"docx": Docx2txtLoader, "pdf": PyPDFLoader, "txt": TextLoader}

In [13]:
def get_loader(filename):
    file_extension = Path(filename).suffix.lstrip(
        "."
    )  # Extract and clean the file extension

    loader_class = loader_classes.get(
        file_extension
    )  # Get the loader class from the dictionary

    if loader_class:
        return loader_class(filename)  # Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

In [16]:
folder_path = Path(
    "./data/CilentoTouristInfo"
)  # A Path to the folder containing the documents

for file_path in folder_path.iterdir():  # B iterate over the files in the path
    if file_path.is_file():  # C Check if it is a file (not a directory)
        try:
            loader = get_loader(
                file_path
            )  # D Instantiate the correct loader for the file
            print(f"Loader for {file_path.name}: {loader}")
            split_and_import(loader)  # E Split and ingest
        except ValueError as e:
            print(e)

Loader for Santa Maria di Castellabate.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x319f89dc0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x319f89dc0>
Loader for Cilento.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x30267f8f0>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x30267f8f0>
Loader for Parco Nazionale del Cilento.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x319f899a0>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x319f899a0>
Loader for Parmenides.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x300de6030>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x300de6030>
Loader for Acciaroli.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader obj

## Vectore Store を検索

In [17]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4)  # four clostest results
print(results)

[Document(id='b1a47a45-0934-4c6b-b1ff-c4b1d747f375', metadata={'source': 'data/Paestum/Paestum-Britannica.docx'}, page_content='Paestum, Greek\xa0Poseidonia, ancient city in southern\xa0Italy\xa0near the west coast, 22 miles (35 km) southeast of modern\xa0Salerno\xa0and 5 miles (8 km) south of the Sele (ancient Silarus) River. Paestum is noted for its splendidly preserved Greek temples.\n\n\n\n\n\nVisit the ruins of the ancient Greek colony of Paestum and discover its history, culture, and society\n\nSee all videos for this article'), Document(id='09372edd-6156-4082-ab01-533387c58668', metadata={'source': 'data/Paestum/Paestum-Britannica.docx'}, page_content='Paestum, Greek\xa0Poseidonia, ancient city in southern\xa0Italy\xa0near the west coast, 22 miles (35 km) southeast of modern\xa0Salerno\xa0and 5 miles (8 km) south of the Sele (ancient Silarus) River. Paestum is noted for its splendidly preserved Greek temples.\n\n\n\n\n\nVisit the ruins of the ancient Greek colony of Paestum and 

In [18]:
len(results)

4

## RAG chain に質問をする

In [35]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.
Answer in Japanese. But do not translate names or file names or path names or the source.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

In [29]:
retriever = vector_db.as_retriever()

In [30]:
from langchain_core.runnables import RunnablePassthrough

question_feeder = RunnablePassthrough()

In [ ]:
chatbot = ChatOllama(model="mistral", temperature=0)

In [32]:
rag_chain = {"context": retriever, "question": question_feeder} | rag_prompt | chatbot

In [33]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [38]:
question = (
    "Where was Poseidonia and who renamed it to Paestum. Also tell me the source."
)
answer = execute_chain(rag_chain, question)
print(answer.content)

 Poseidonia (あなたの名前はPaestumに変更されました) は、イタリアの南部にあり、現代のサレルノから22マイル(35キロ)東南、セレ川(古代のシラルス川)川から5マイル(8キロ)離れています。 この都市は、古代ギリシアのテンプルが崩壊しなく保存されていることに注目されています。

ソース: Encyclopaedia Britannica (データ/Paestum/Paestum-Britannica.docx)


In [39]:
question = "And then, what did they do? Also tell me the source"
answer = execute_chain(rag_chain, question)
print(answer.content)

 文書ID '77198fc8-fce1-46c2-ab85-0ad9cdb4a463' によると、アテナの周りで赤色の形を持つ陶器（エリュトロモルファ）は、古 Greece で使用された。このスタイルは、520 BC 近くにアテナで開発され、黒色の背景に赤色またはオレンジ色の人物や詳細を描いている。このスタイルは、前に主要なスタイルとして存在した黒色陶器（ブラックフィギャー）を代替するという modern name であり、それに対し、背景が黒く、人物や詳細は赤く描かれる black-figure style は、前のスタイルであった。このスタイルは、アテカ以外の重要な製造地は、南イタリアであり、その他の部分 Greece にも採用された。エトルリアは、非ギリシャ世界での重要な製造地となった。Attic red-figure vases were exported throughout Greece and beyond. For a long time, they dominated the market for fine ceramics.

Source: https://en.wikipedia.org/wiki/Red-figure_pottery (Red-figure pottery)


In [47]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible. You must answer in Japanese. But do not translate names or file names or path names or the source.",
        ),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOllama(model="mistral", temperature=0)
chat_history_memory = ChatMessageHistory()


def get_messages(x):
    return chat_history_memory.messages


rag_chain = (
    {
        "retrieved_context": retriever,
        "question": question_feeder,
        "chat_history_messages": RunnableLambda(get_messages),
    }
    | rag_prompt
    | chatbot
)


def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f"Full chat message history: {chat_history_memory.messages}\n\n")
    return answer

In [48]:
question = (
    # "Where was Poseidonia and who renamed it to Paestum? Also tell me the source."
    "ポセイドニアはどこにあり、誰がそれをパエストゥムと改名しましたか？また、その情報源も教えてください。"
)
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='ポセイドニアはどこにあり、誰がそれをパエストゥムと改名しましたか？また、その情報源も教えてください。', additional_kwargs={}, response_metadata={}), AIMessage(content=' ポセイドニア（Poseidonia）はカンペリオ（Campania）地方のマリナ・ディ・カメロタ（Marina di Camerota）にあります。古代の時代、ローマ人が建てた都市で、後でパエストゥム（Paestum）という名前で知られるようになりました。情報源は、Wikipediaの「Roman Catholic Diocese of Pesto」ページです。', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2025-10-29T04:18:35.964817Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11782158375, 'load_duration': 2242593583, 'prompt_eval_count': 1262, 'prompt_eval_duration': 2804087625, 'eval_count': 131, 'eval_duration': 4618474621, 'model_name': 'mistral'}, id='run--17a6eb18-0108-41cf-a91c-aebe3bf34054-0', usage_metadata={'input_tokens': 1262, 'output_tokens': 131, 'total_tokens': 1393})]


 ポセイドニア（Poseidonia）はカンペリオ（Campania）地方のマリナ・ディ・カメロタ（Marina di Camerota）にあります。古代の時代、ローマ人が建てた都市で、後でパエストゥム（Paestum）という名前で知られるようになりました。情報源は、Wikipediaの「Roman Catholic Diocese of 

In [49]:
# question = "And then what did they do? Also tell me the source"
question = "それから彼らは何をしましたか？また、その情報源も教えてください"
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='ポセイドニアはどこにあり、誰がそれをパエストゥムと改名しましたか？また、その情報源も教えてください。', additional_kwargs={}, response_metadata={}), AIMessage(content=' ポセイドニア（Poseidonia）はカンペリオ（Campania）地方のマリナ・ディ・カメロタ（Marina di Camerota）にあります。古代の時代、ローマ人が建てた都市で、後でパエストゥム（Paestum）という名前で知られるようになりました。情報源は、Wikipediaの「Roman Catholic Diocese of Pesto」ページです。', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2025-10-29T04:18:35.964817Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11782158375, 'load_duration': 2242593583, 'prompt_eval_count': 1262, 'prompt_eval_duration': 2804087625, 'eval_count': 131, 'eval_duration': 4618474621, 'model_name': 'mistral'}, id='run--17a6eb18-0108-41cf-a91c-aebe3bf34054-0', usage_metadata={'input_tokens': 1262, 'output_tokens': 131, 'total_tokens': 1393}), HumanMessage(content='それから彼らは何をしましたか？また、その情報源も教えてください', additional_kwargs={}, response_metadata={}), AIMessage(content=' ポセイドニア（Poseidonia）はローマ人が建てた都市で、後でパエストゥム（Paestum）という名